In [ ]:
!git clone https://github.com/had-A42/Actions-Speak-Louder-than-Words-research.git
%cd Actions-Speak-Louder-than-Words-research
!git checkout hstu_hadhad

In [ ]:
!pip3 install -r requirements.txt

In [10]:
import sys
from pathlib import Path

for project_root in (Path.cwd(), *Path.cwd().parents):
    if (project_root / "src").exists():
        sys.path.insert(0, str(project_root))
        break

import pandas as pd
from src.data.loaders import (load_yambda, split_and_reindex)

In [ ]:
config = {
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "../../../tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
}

In [5]:
yambda_df = load_yambda(config["yambda-retrieval"]["interactions_path"], config=config["yambda-retrieval"])

In [6]:
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda-retrieval"])

In [7]:
datasets = {
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    }
}

In [11]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7473676,748537,8222213,0.909,0.091,79059,53927,163448


In [ ]:
import time

import torch

from src.HSTU.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.HSTU.models.hstu import HSTUModel


## HSTU Yambda experiment

Запуск HSTU-пайплайна на Yambda

In [ ]:
base_hstu_config = HSTUExperimentConfig(
    max_seq_len=config["yambda-retrieval"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=4,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=2,
    num_heads=2,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias=True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

sl_hstu_config = HSTUExperimentConfig(
    **{
        **base_hstu_config.__dict__,
        "stochastic_length_alpha": 1.5,
        "stochastic_length_sampling": "random_subsequence",
    }
)

baseline_train_loader, baseline_eval_loader, baseline_train_dataset, baseline_eval_dataset, baseline_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=base_hstu_config.max_seq_len,
    train_batch_size=base_hstu_config.train_batch_size,
    eval_batch_size=base_hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=base_hstu_config.num_workers,
)

sl_train_loader, sl_eval_loader, sl_train_dataset, sl_eval_dataset, sl_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=sl_hstu_config.max_seq_len,
    train_batch_size=sl_hstu_config.train_batch_size,
    eval_batch_size=sl_hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=sl_hstu_config.num_workers,
    stochastic_length_alpha=sl_hstu_config.stochastic_length_alpha,
    stochastic_length_sampling=sl_hstu_config.stochastic_length_sampling,
)

hstu_config = sl_hstu_config
hstu_train_loader = sl_train_loader
hstu_eval_loader = sl_eval_loader
hstu_train_dataset = sl_train_dataset
hstu_eval_dataset = sl_eval_dataset
hstu_targets = sl_targets

{
    "baseline": (len(baseline_train_dataset), len(baseline_eval_dataset), len(baseline_targets)),
    "stochastic_length": (len(sl_train_dataset), len(sl_eval_dataset), len(sl_targets)),
    "sl_alpha": sl_hstu_config.stochastic_length_alpha,
    "sl_sampling": sl_hstu_config.stochastic_length_sampling,
}


In [ ]:
def build_hstu_model_and_optimizer(experiment_config):
    model = HSTUModel(
        num_items=yambda_data_description["n_items"],
        embedding_dim=experiment_config.embedding_dim,
        max_seq_len=experiment_config.max_seq_len,
        num_blocks=experiment_config.num_blocks,
        num_heads=experiment_config.num_heads,
        linear_dim=experiment_config.linear_dim,
        attention_dim=experiment_config.attention_dim,
        num_negatives=experiment_config.num_negatives,
        softmax_temperature=experiment_config.softmax_temperature,
        sampling_strategy=experiment_config.sampling_strategy,
        user_embedding_norm=experiment_config.user_embedding_norm,
        l2_norm_embeddings=experiment_config.l2_norm_embeddings,
        l2_norm_eps=experiment_config.l2_norm_eps,
        item_id_offset=experiment_config.item_id_offset,
        input_dropout_rate=experiment_config.input_dropout_rate,
        linear_dropout_rate=experiment_config.linear_dropout_rate,
        attn_dropout_rate=experiment_config.attn_dropout_rate,
        enable_relative_attention_bias=experiment_config.enable_relative_attention_bias,
        relative_attention_num_buckets=experiment_config.relative_attention_num_buckets,
    )
    model.to(experiment_config.device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=experiment_config.learning_rate,
        weight_decay=experiment_config.weight_decay,
    )
    return model, optimizer


baseline_model, baseline_optimizer = build_hstu_model_and_optimizer(base_hstu_config)
sl_model, sl_optimizer = build_hstu_model_and_optimizer(sl_hstu_config)

hstu_model = sl_model
hstu_optimizer = sl_optimizer


In [ ]:
params_sum = 0
trainable_sum = 0

for name, module in sl_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")


In [ ]:
def train_hstu_with_epoch_timing(
    model,
    optimizer,
    train_loader,
    eval_loader,
    targets,
    experiment_config,
    run_name,
):
    losses = []
    eval_history = []
    epoch_times = []

    for epoch in range(experiment_config.num_epochs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()

        epoch_losses = train_hstu(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            num_epochs=1,
            device=experiment_config.device,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - start_time

        epoch_metrics, _ = evaluate_hstu(
            model=model,
            eval_loader=eval_loader,
            targets=targets,
            catalog_size=yambda_data_description["n_items"],
            topk=experiment_config.topk,
            device=experiment_config.device,
            filter_seen=experiment_config.filter_seen,
        )

        losses.extend(epoch_losses)
        eval_history.append(epoch_metrics)
        epoch_times.append(epoch_time)
        print(
            f"{run_name} epoch {epoch + 1}/{experiment_config.num_epochs}: "
            f"train_time={epoch_time:.2f}s, "
            f"loss={epoch_losses[-1]:.4f}"
        )

    return losses, eval_history, epoch_times


baseline_losses, baseline_eval_history, baseline_epoch_times = train_hstu_with_epoch_timing(
    model=baseline_model,
    optimizer=baseline_optimizer,
    train_loader=baseline_train_loader,
    eval_loader=baseline_eval_loader,
    targets=baseline_targets,
    experiment_config=base_hstu_config,
    run_name="baseline",
)

sl_losses, sl_eval_history, sl_epoch_times = train_hstu_with_epoch_timing(
    model=sl_model,
    optimizer=sl_optimizer,
    train_loader=sl_train_loader,
    eval_loader=sl_eval_loader,
    targets=sl_targets,
    experiment_config=sl_hstu_config,
    run_name="stochastic_length",
)

losses = sl_losses
eval_history = sl_eval_history
epoch_times = sl_epoch_times

{
    "baseline_losses": baseline_losses,
    "baseline_epoch_times": baseline_epoch_times,
    "baseline_eval_history": baseline_eval_history,
    "stochastic_length_losses": sl_losses,
    "stochastic_length_epoch_times": sl_epoch_times,
    "stochastic_length_eval_history": sl_eval_history,
}


In [20]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.18243180595990877,
 'recall': 0.03772155538895536,
 'ndcg': 0.03335553624752489,
 'coverage': 0.22319636826391268}

In [21]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.395738683776216,
 'recall': 0.08343633669724367,
 'ndcg': 0.04806770518668592,
 'coverage': 0.4596140668591846}

In [22]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.499174810391826,
 'recall': 0.12554879618608492,
 'ndcg': 0.060777557568382455,
 'coverage': 0.594207331995497}

In [23]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.5994399836816438,
 'recall': 0.1856030194230913,
 'ndcg': 0.07707611246580004,
 'coverage': 0.7356896382947482}

In [12]:
import torch

torch.save(hstu_model.state_dict(), "yambda-retrieval-model.pt")

NameError: name 'hstu_model' is not defined

In [ ]:
checkpoint = {
    "epoch": hstu_config.num_epochs,
    "model_state_dict": hstu_model.state_dict(),
    "optimizer_state_dict": hstu_optimizer.state_dict(),
    "loss": losses,
}

torch.save(checkpoint, "checkpoint.pt")